# Step 2.5: Compare Vectors Across Personas

Core analysis: how similar are the steering vectors for the same trait when
extracted under different personas along the assistant axis?

Key questions:
- Do adjacent personas (e.g. mild assistant vs full assistant) have more similar vectors?
- Does similarity decay monotonically with axis distance?
- Which traits are most/least persona-dependent?
- How much of each vector is shared vs persona-specific?

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

from persona_steering.config import VECTORS_DIR, Trait, AXIS_PERSONA_ORDER
from persona_steering.analysis import (
    compare_vectors,
    build_transfer_matrix,
    build_per_trait_transfer,
    decompose_shared_specific,
    compute_curvature,
)
from persona_steering.personas import load_all_personas, load_axis_personas
from persona_steering.utils import load_pickle, ensure_output_dirs

ensure_output_dirs()
all_vectors = load_pickle(VECTORS_DIR / "all_vectors.pkl")
all_personas = load_all_personas()
axis_personas = load_axis_personas()

all_slugs = [p.slug for p in all_personas]
axis_slugs = [p.slug for p in axis_personas]
axis_positions = {p.slug: p.position for p in axis_personas}
traits = list(Trait)

print(f"All personas: {all_slugs}")
print(f"Axis personas: {axis_slugs}")
print(f"Traits: {[t.value for t in traits]}")

In [ ]:
# Pick representative layer
sample_trait = traits[0]
sample_layers = sorted(all_vectors[all_slugs[0]][sample_trait].keys())
mid_layer = sample_layers[len(sample_layers) // 2]
print(f"Using layer {mid_layer} for comparison")

# Pairwise comparison for axis personas per trait
for trait in traits:
    print(f"\n--- {trait.value} ---")
    for i, pa in enumerate(axis_slugs):
        for j, pb in enumerate(axis_slugs):
            if j <= i:
                continue
            va = all_vectors[pa][trait][mid_layer]
            vb = all_vectors[pb][trait][mid_layer]
            comp = compare_vectors(va, vb)
            dist = abs(axis_positions[pa] - axis_positions[pb])
            print(f"  {pa} vs {pb} (dist={dist:.1f}): cos={comp.cosine_sim:.4f}")

In [ ]:
# Transfer matrix for ALL personas (including base model)
matrix_all = build_transfer_matrix(all_vectors, all_slugs, traits, mid_layer)
all_names = [p.name for p in all_personas]

fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(matrix_all, annot=True, fmt=".3f", xticklabels=all_names,
            yticklabels=all_names, cmap="RdYlBu_r", vmin=0, vmax=1,
            ax=ax, square=True, linewidths=0.5)
ax.set_title(f"Steering Vector Similarity — All Personas (Layer {mid_layer})")
plt.tight_layout()
plt.savefig("../outputs/figures/transfer_matrix_all.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Transfer matrix for axis personas only — should show gradient structure
matrix_axis = build_transfer_matrix(all_vectors, axis_slugs, traits, mid_layer)
axis_labels = [f"{p.name}\n({p.position:+.1f})" for p in axis_personas]

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(matrix_axis, annot=True, fmt=".3f", xticklabels=axis_labels,
            yticklabels=axis_labels, cmap="viridis", vmin=0, vmax=1,
            ax=ax, square=True, linewidths=0.5)
ax.set_title(f"Axis Persona Transfer Matrix (Layer {mid_layer})")
plt.tight_layout()
plt.savefig("../outputs/figures/transfer_matrix_axis.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Per-trait transfer matrices (axis personas)
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for ax, trait in zip(axes.flat, traits):
    m = build_per_trait_transfer(all_vectors, axis_slugs, trait, mid_layer)
    sns.heatmap(m, annot=True, fmt=".3f", xticklabels=[p.name for p in axis_personas],
                yticklabels=[p.name for p in axis_personas], cmap="viridis",
                vmin=0, vmax=1, ax=ax, square=True, linewidths=0.5)
    ax.set_title(trait.value.title())

plt.suptitle(f"Per-Trait Transfer Matrices (Layer {mid_layer})", fontsize=14)
plt.tight_layout()
plt.savefig("../outputs/figures/per_trait_transfer.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Cosine similarity vs axis distance — the key decay curve
fig, ax = plt.subplots(figsize=(8, 5))
colors = plt.cm.tab10(np.linspace(0, 1, len(traits)))

for trait, color in zip(traits, colors):
    dists, sims = [], []
    for i, pa in enumerate(axis_slugs):
        for j, pb in enumerate(axis_slugs):
            if j <= i:
                continue
            va = all_vectors.get(pa, {}).get(trait, {}).get(mid_layer)
            vb = all_vectors.get(pb, {}).get(trait, {}).get(mid_layer)
            if va and vb:
                from persona_steering.utils import cosine_similarity
                dists.append(abs(axis_positions[pa] - axis_positions[pb]))
                sims.append(cosine_similarity(va.vector, vb.vector))
    ax.scatter(dists, sims, color=color, label=trait.value, alpha=0.7, s=40)

ax.set_xlabel("Axis Distance Between Personas")
ax.set_ylabel("Cosine Similarity of Steering Vectors")
ax.set_title("Steering Vector Similarity vs Persona Distance")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("../outputs/figures/similarity_vs_distance.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Shared vs specific decomposition per trait (axis personas)
for trait in traits:
    vecs = {slug: all_vectors[slug][trait][mid_layer] for slug in axis_slugs
            if trait in all_vectors.get(slug, {}) and mid_layer in all_vectors[slug].get(trait, {})}
    if len(vecs) >= 2:
        decomp = decompose_shared_specific(vecs)
        print(f"\n{trait.value}: variance_explained={decomp.variance_explained:.4f}")
        for slug in axis_slugs:
            if slug in decomp.shared_magnitudes:
                print(f"  {slug}: shared_mag={decomp.shared_magnitudes[slug]:.4f}, "
                      f"specific_mag={decomp.specific_magnitudes[slug]:.4f}")

In [ ]:
# Curvature / decay analysis
# Build distance matrix from axis positions
n = len(axis_slugs)
dist_matrix = np.zeros((n, n))
for i, pa in enumerate(axis_slugs):
    for j, pb in enumerate(axis_slugs):
        dist_matrix[i, j] = abs(axis_positions[pa] - axis_positions[pb])

curvature = compute_curvature(matrix_axis, dist_matrix)
print("\nDecay analysis (axis personas):")
for k, v in curvature.items():
    print(f"  {k}: {v:.4f}")